In [ ]:
# Importer des bibliothèques standard
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os
import math
import seaborn as sns
import re

# Importer des bibliothèques pour le prétraitement des données
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import confusion_matrix, accuracy_score

# Importation de bibliothèques pour les mesures de similarité et de distance
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Importation de bibliothèques pour l'analyse de texte
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# Importer les bibliothèques pour Colab et la gestion des fichiers
from google.colab import drive
from google.colab import files

# Outils divers
import sklearn

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Charger les fichiers
interactions = pd.read_csv('https://raw.githubusercontent.com/Vane-s/UNIL_Rolex/main/interactions_train.csv')
items = pd.read_csv("https://raw.githubusercontent.com/Vane-s/UNIL_Rolex/main/items.csv")

# Afficher les premières lignes de chaque ensemble de données
display(interactions.head())
display(items.head())

,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09


,Title,Author,ISBN Valid,Publisher,Subjects,i
0,Classification décimale universelle : édition ...,NaN,9782871303336; 2871303339,Ed du CEFAL,Classification décimale universelle; Indexatio...,0
1,Les interactions dans l'enseignement des langu...,"Cicurel, Francine, 1947-",9782278058327; 2278058320,Didier,didactique--langue étrangère - enseignement; d...,1
2,Histoire de vie et recherche biographique : pe...,NaN,2343190194; 9782343190198,L'Harmattan,Histoires de vie en sociologie; Sciences socia...,2
3,Ce livre devrait me permettre de résoudre le c...,"Mazas, Sylvain, 1980-",9782365350020; 236535002X; 9782365350488; 2365...,Vraoum!,Moyen-Orient; Bandes dessinées autobiographiqu...,3
4,Les années glorieuses : roman /,"Lemaitre, Pierre, 1951-",9782702180815; 2702180817; 9782702183618; 2702...,Calmann-Lévy,France--1945-1975; Roman historique; Roman fra...,4


In [ ]:
# Renommer les colonnes du dataset interactions
interactions.rename(columns={
    'u': 'user_id',        # Colonne utilisateur
    'i': 'item_id',        # Colonne identifiant du livre
    't': 'timestamp'       # Colonne horodatage
}, inplace=True)

# Renommer les colonnes du dataset items
items.rename(columns={
    'Title': 'title',          # Titre du livre
    'Author': 'author',        # Auteur
    'ISBN Valid': 'isbn',      # ISBN
    'Publisher': 'publisher',  # Éditeur
    'Subjects': 'subjects',    # Catégories/thèmes
    'i': 'item_id'             # Identifiant du livre
}, inplace=True)

# Vérifier les colonnes après renommage
print("Colonnes interactions:", interactions.columns)
print("Colonnes items:", items.columns)

Colonnes interactions: Index(['user_id', 'item_id', 'timestamp'], dtype='object')
Colonnes items: Index(['title', 'author', 'isbn', 'publisher', 'subjects', 'item_id'], dtype='object')


In [ ]:
# Nettoyage du DataFrame items
def clean_text(text):
    if pd.isna(text):
        return ""
    # Supprimer les caractères spéciaux et les espaces supplémentaires
    text = re.sub(r'[^\w\s]', '', text)
    # Convertir en minuscules
    text = text.lower()
    # Enlever les espaces supplémentaires
    text = " ".join(text.split())
    return text

items['title'] = items['title'].apply(clean_text)
items['author'] = items['author'].apply(clean_text)
items['isbn'] = items['isbn'].apply(clean_text)
items['subjects'] = items['subjects'].apply(clean_text)
items['publisher'] = items['publisher'].apply(clean_text)

In [ ]:
#Tri des données par utilisateur et par timestamp
interactions = interactions.sort_values(["user_id", "timestamp"])

#Calcul du rang proportionnel par utilisateur
interactions["pct_rank"] = interactions.groupby("user_id")["timestamp"].rank(pct=True, method="dense")
interactions.reset_index(inplace=True, drop=True)

#Calcul de l'atténuation temporelle
time_decay = np.exp(-(interactions["timestamp"].max() - interactions["timestamp"]) / (365 * 24 * 60 * 60))
interactions["time_decay"] = time_decay

n_users = interactions["user_id"].max() + 1
n_items = interactions["item_id"].max() + 1
print("Nombre d'utilisateurs:", n_users)
print("Nombre d'items:", n_items)


Nombre d'utilisateurs: 7838
Nombre d'items: 15291


In [ ]:
# Définition de la fonction pour créer la matrice utilisateur-item
def create_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["user_id"].values, data["item_id"].values] = data["time_decay"].values
    return data_matrix

# Créer la matrice utilisateur-item avec la pondération temporelle
train_data_matrix = create_data_matrix(interactions, n_users, n_items)

# Calcul de la similarité entre items
item_similarity = cosine_similarity(train_data_matrix.T)

# Calcul de la similarité entre utilisateurs
user_similarity = cosine_similarity(train_data_matrix)


# Définition de la fonction pour prédire avec la similarité des items
def item_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T

# Générer les prédictions basées sur la similarité des items
item_prediction = item_predict(train_data_matrix, item_similarity)


# Définition de la fonction pour prédire avec la similarité des utilisateurs
def user_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred

# Générer les prédictions basées sur la similarité des utilisateurs
user_prediction = user_predict(train_data_matrix, user_similarity)

In [ ]:
# Charger un modèle pré-entraîné basé sur DistilRoBERTa
model = SentenceTransformer('all-distilroberta-v1')

# Create 'combined_text' column by concatenating relevant columns
items['combined_text'] = items['title'].astype(str) + ' ' + items['author'].astype(str) + ' ' + items['subjects'].astype(str)

# Générer les embeddings pour les items en utilisant la colonne combinée
combined_item_embeddings = model.encode(items['combined_text'].tolist(), show_progress_bar=True)


Batches:   0%|          | 0/478 [00:00<?, ?it/s]

In [ ]:
# Calculer la similarité entre les items en utilisant les embeddings textuels
text_similarity = cosine_similarity(combined_item_embeddings)

# Prédiction des items recommandés avec la similarité combinée
def item_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T

item_prediction_combined = item_predict(train_data_matrix, text_similarity)


In [ ]:
# Initialisation du MinMaxScaler pour normalisation
scaler = MinMaxScaler()

# Normalisation des prédictions
user_prediction_normalized = scaler.fit_transform(user_prediction.reshape(-1, 1)).flatten()
item_prediction_normalized = scaler.fit_transform(item_prediction.reshape(-1, 1)).flatten()
item_prediction_combined_normalized = scaler.fit_transform(item_prediction_combined.reshape(-1, 1)).flatten()

In [ ]:
# Définition des poids pour le modèle hybride
x = 0.4
y = 0.4
z = 1 - x - y

model_hybrid = x * user_prediction + y * item_prediction + z * item_prediction_combined

In [ ]:
# Générer les 10 meilleures recommandations avec une logique unifiée
recommendations = []
for user_id in range(model_hybrid.shape[0]):  # utiliser model_hybrid comme dans la fonction
    top_10 = np.argsort(model_hybrid[user_id, :])[-10:][::-1]  # Identique à la logique de top_k_recommendations_funct
    recommendations.append(" ".join(map(str, top_10)))

# Créer un DataFrame avec les recommandations
import pandas as pd  # Importation de pandas pour manipuler les DataFrames
submission_df = pd.DataFrame({
    "user_id": range(model_hybrid.shape[0]),  # Utilise model_hybrid pour correspondre au nombre d'utilisateurs
    "recommendation": recommendations  # Recommandations formatées
})

# Sauvegarder le fichier CSV dans l'environnement Colab
submission_file = "predictions_submission.csv"  # Nom du fichier
submission_df.to_csv(submission_file, index=False)  # Sauvegarde sans index

# Télécharger le fichier localement
from google.colab import files  # Importation pour téléchargement local
files.download(submission_file)

# Vérifier le résultat
print(submission_df.head(10))  # Afficher les 10 premières lignes pour vérifier les recommandations

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   user_id                                     recommendation
0        0                         21 24 1 20 17 2 7 23 22 13
1        1                      39 31 38 37 33 35 36 32 30 34
2        2                      94 92 80 76 79 77 50 57 60 82
3        3            157 132 145 140 162 151 155 168 166 172
4        4            192 204 203 202 205 195 200 197 207 194
5        5            221 223 222 212 224 219 218 220 217 216
6        6    229 230 231 232 3862 2829 13950 7568 13988 5638
7        7            246 240 250 249 245 247 236 243 242 248
8        8  258 257 256 14986 4385 10616 14417 13371 14483...
9        9    262 264 261 263 1556 4381 13709 11774 4023 1263
